# Collect gesture samples
Records webcam frames for one gesture label at a time, extracts landmarks with MediaPipe, and appends them to `data/annotations/<label>.npy`.

In [ ]:
import sys
sys.path.append('..')

import os
import cv2
import numpy as np

from src.components.hand_tracker import HandTracker

DATA_DIR = '../data/annotations'
os.makedirs(DATA_DIR, exist_ok=True)

In [ ]:
def collect(label: str, num_samples: int = 200, camera_id: int = 0):
    """Show a preview window; hold the gesture steady while it records."""
    tracker = HandTracker()
    cap = cv2.VideoCapture(camera_id)
    samples = []

    while len(samples) < num_samples:
        ok, frame = cap.read()
        if not ok:
            continue

        frame = cv2.flip(frame, 1)

        hands = tracker.process(frame)
        if hands:
            samples.append(hands[0])
            cv2.putText(frame, f'{label}: {len(samples)}/{num_samples}', (10, 30),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

        cv2.imshow('collect', frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    tracker.close()
    cv2.destroyAllWindows()

    out_path = f'{DATA_DIR}/{label}.npy'
    arr = np.array(samples, dtype=np.float32)

    if os.path.exists(out_path):
        arr = np.concatenate([np.load(out_path), arr], axis=0)

    np.save(out_path, arr)
    print(f'saved {arr.shape[0]} samples to {out_path}')

In [ ]:
# collect('open_palm')
# collect('close_palm')
# collect('one_finger_up')
# collect('one_finger_down')
# collect('two_finger_up')
# collect('two_finger_down')
# collect('pinch')